## Chat Models - Tagging Documents

In [ ]:
%pip install langchain langchain-openai langchain-community langchain-text-splitters lxml beautifulsoup4 langchain-classic pandas --upgrade

  Using cached lxml-6.0.2-cp312-cp312-macosx_10_13_universal2.whl.metadata (3.6 kB)
Using cached lxml-6.0.2-cp312-cp312-macosx_10_13_universal2.whl (8.7 MB)
  Attempting uninstall: lxml
    Found existing installation: lxml 5.4.0
    Uninstalling lxml-5.4.0:
      Successfully uninstalled lxml-5.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sammo 0.3.3 requires lxml<6.0,>=5.3, but you have lxml 6.0.2 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import getpass
import os
import warnings

# Suppress LangChain deprecation warnings (we're intentionally using legacy chains)
warnings.filterwarnings("ignore", category=DeprecationWarning, module="langchain")

# Set USER_AGENT for web requests
os.environ["USER_AGENT"] = "langchain-tagging-notebook"

# Set up your OpenAI client
if not os.getenv("OPENAI_API_KEY"):
    secret_key = getpass.getpass("Enter your OpenAI API key: ")
    os.environ["OPENAI_API_KEY"] = secret_key

In [12]:
# fixes a bug with asyncio and jupyter
import nest_asyncio

nest_asyncio.apply()

In [13]:
from langchain_community.document_loaders.sitemap import SitemapLoader
from langchain_openai.chat_models import ChatOpenAI
from langchain_classic.chains import create_tagging_chain, create_tagging_chain_pydantic
import pandas as pd

In [14]:
sitemap_loader = SitemapLoader(web_path="https://understandingdata.com/sitemap.xml")
sitemap_loader.requests_per_second = 5
docs = sitemap_loader.load()

Fetching pages: 100%|##########| 268/268 [00:09<00:00, 27.02it/s]


In [15]:
# Schema
schema = {
    "properties": {
        "sentiment": {"type": "string" },
        "aggressiveness": {"type": "integer"},
        "primary_topic": {"type": "string", "description": "The main topic of the document."},
    },
     "required": ["primary_topic", "sentiment", "aggressiveness"],
}

# LLM
llm = ChatOpenAI(temperature=0)
chain = create_tagging_chain(schema, llm, output_key='output')

In [16]:
results = []

# Remove the 0:10 to run on all documents:
for doc in docs[0:10]:
    print(doc)
    chain_result = chain.invoke({'input': doc.page_content})
    results.append(chain_result['output'])

page_content='James Phoenix | AI Engineer & O'Reilly Author | Production AI SystemsJames PhoenixAboutProjectsServicesPostsContactAI Infrastructure EngineerJames PhoenixI build AI systems that work in production, not demos that break under load.Book a ConsultationView ProjectsGitHubLinkedIn350K+latest project code size306K+developers taught10K+O'Reilly books sold45+bootcamps deliveredServicesWhat I BuildProduction-grade systems with observability, reliability, and performance at scale.SaaS ApplicationsFull-stack SaaS development from idea to production. Including auth, payments, analytics, and automated testing.Learn moreLLM EngineeringBuilding custom AI solutions and integrating LLMs into existing applications. Specializing in RAG, fine-tuning, and prompt engineering.Learn moreReact DevelopmentModern web applications built with Next.js, TypeScript, and Tailwind. Focus on performance, accessibility, and user experience.Learn morePython DevelopmentBuilding scalable backend services, data

In [7]:
results

[{'sentiment': 'positive',
  'aggressiveness': 3,
  'primary_topic': 'AI Engineer'},
 {'sentiment': 'positive',
  'aggressiveness': 3,
  'primary_topic': 'AI Engineer'},
 {'sentiment': 'neutral', 'aggressiveness': 0, 'primary_topic': 'Contact'},
 {'sentiment': 'positive',
  'aggressiveness': 0,
  'primary_topic': 'Production-Grade Engineering'},
 {'sentiment': 'positive', 'aggressiveness': 3, 'primary_topic': 'AI'},
 {'sentiment': 'neutral', 'aggressiveness': 0, 'primary_topic': 'AI Agents'},
 {'sentiment': 'positive',
  'aggressiveness': 0,
  'primary_topic': 'Data Engineering Services'},
 {'sentiment': 'positive',
  'aggressiveness': 3,
  'primary_topic': 'React Software Development'},
 {'sentiment': 'positive',
  'aggressiveness': 0,
  'primary_topic': 'Python software development'},
 {'sentiment': 'positive',
  'aggressiveness': 0,
  'primary_topic': 'Python software development'}]

In [8]:
df = pd.DataFrame(results)

In [9]:
docs[0].metadata

{'source': 'https://understandingdata.com/',
 'loc': 'https://understandingdata.com/',
 'lastmod': '2026-03-17T14:35:00.683Z',
 'changefreq': 'monthly',
 'priority': '1.0'}

In [10]:
# Combine the URLs with the results
df['url'] = [doc.metadata['source'] for doc in docs[0:10]]

In [12]:
df

,sentiment,aggressiveness,primary_topic,url
0,positive,3,AI Engineer,https://understandingdata.com/
1,positive,3,AI Engineer,https://understandingdata.com/about/
2,neutral,0,Contact,https://understandingdata.com/contact/
3,positive,3,Production-Grade Engineering,https://understandingdata.com/services/
4,positive,3,AI,https://understandingdata.com/projects/
5,neutral,0,AI Agents,https://understandingdata.com/posts/
6,positive,3,Data Engineering Services,https://understandingdata.com/services/data-en...
7,positive,0,React Software Development,https://understandingdata.com/services/react-s...
8,positive,0,Python software development,https://understandingdata.com/services/python-...
9,positive,0,Python software development,https://understandingdata.com/services/python-...
